In [ ]:
import os

import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
# Path-neutral configuration. Set these two environment variables for your
# local or cluster data locations before running the notebook.
DATA_ROOT = os.environ["AMAZON_DATA_DIR"]
OUTPUT_ROOT = os.environ["AMAZON_OUTPUT_DIR"]

HISTORIC_BASE_PATH = os.path.join(
    DATA_ROOT, "intermediate", "02_bioclimatic_variables", "historical"
)
FUTURE_BASE_PATH = os.path.join(
    DATA_ROOT, "intermediate", "02_bioclimatic_variables", "future"
)
PLOT_DIR = os.path.join(
    OUTPUT_ROOT, "08_figures", "appendix", "bioclimatic_variables", "plots"
)
os.makedirs(PLOT_DIR, exist_ok=True)


def load_historic_bioclim(start_year=1980, end_year=2014):
    historic_path = os.path.join(
        HISTORIC_BASE_PATH,
        f"bioclimatic_variables_historic_ERA5_{start_year}_{end_year}.nc",
    )
    return xr.open_dataset(historic_path, engine="netcdf4")


def load_future_bioclim(
    ssp,
    start_year,
    end_year,
    tipping=False,
    tipping_type="deforestation",
):
    scenario_parts = ("tip", tipping_type) if tipping else ("notip",)
    future_path = os.path.join(
        FUTURE_BASE_PATH,
        ssp,
        *scenario_parts,
        f"bioclim_vars_ETresid_{start_year}_{end_year}.nc",
    )
    return xr.open_dataset(future_path, engine="netcdf4")


def load_amazon_mask(mask_path):
    amazon_mask = rioxarray.open_rasterio(mask_path).squeeze(drop=True)
    return amazon_mask.rename({"x": "longitude", "y": "latitude"})


def apply_amazon_mask(ds, amazon_mask):
    return ds.sel(
        longitude=amazon_mask.longitude,
        latitude=amazon_mask.latitude,
    ).where(amazon_mask.notnull())


amazon_mask_path = os.path.join(
    DATA_ROOT, "intermediate", "01_c_noresm2", "amazon_mask", "amazon_mask.tif"
)
amazon_mask = load_amazon_mask(amazon_mask_path)

historic_bioclim = load_historic_bioclim()
historic_bioclim_amazon = apply_amazon_mask(historic_bioclim, amazon_mask)

periods = [(2030, 2044), (2050, 2069), (2080, 2099)]
ssp = "ssp245"

future_notip = {}
future_tip = {}
for start_year, end_year in periods:
    key = f"{start_year}_{end_year}"
    future_notip[key] = apply_amazon_mask(
        load_future_bioclim(ssp, start_year, end_year, tipping=False),
        amazon_mask,
    )
    # Temperature is identical in both scenarios; precipitation-related
    # variables retain the Monte-Carlo sample dimension and are averaged here.
    future_tip[key] = apply_amazon_mask(
        load_future_bioclim(ssp, start_year, end_year, tipping=True).mean(dim="sample"),
        amazon_mask,
    )

In [ ]:
# Use historic MAT as reference
mat_hist = historic_bioclim_amazon["MAT"]

# Boolean mask where MAT is valid
valid_mask = mat_hist.notnull()

# Get latitude and longitude bounds
lat_vals = mat_hist.latitude.where(valid_mask.any("longitude"), drop=True)
lon_vals = mat_hist.longitude.where(valid_mask.any("latitude"), drop=True)

lat_min = float(lat_vals.min())
lat_max = float(lat_vals.max())
lon_min = float(lon_vals.min())
lon_max = float(lon_vals.max())

print("Spatial bounds used for plotting:")
print(f"Latitude:  {lat_min:.2f} to {lat_max:.2f}")
print(f"Longitude: {lon_min:.2f} to {lon_max:.2f}")


In [ ]:
def subset_to_valid_extent(ds, lat_min, lat_max, lon_min, lon_max):
    return ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )


historic_bioclim_amazon = subset_to_valid_extent(
    historic_bioclim_amazon,
    lat_min, lat_max,
    lon_min, lon_max
)

mat_hist = historic_bioclim_amazon["MAT"]


for key in future_notip:
    future_notip[key] = subset_to_valid_extent(
        future_notip[key],
        lat_min, lat_max,
        lon_min, lon_max
    )

for key in future_tip:
    future_tip[key] = subset_to_valid_extent(
        future_tip[key],
        lat_min, lat_max,
        lon_min, lon_max
    )



In [ ]:
def percentage_difference(future, reference):
    """Calculate (future - reference) / reference * 100."""
    return (future - reference) / reference * 100


def make_label_axis(ax, label):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylabel(
        label,
        rotation=90,
        fontsize=16,
        fontweight="bold",
        labelpad=2,
        va="center",
    )

def add_panel_label(ax, label):
    """Add an SI panel letter in the upper-left corner of a map axis."""
    ax.text(
        0.02,
        0.98,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=16,
        fontweight="bold",
        color="black",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
        zorder=10,
    )


In [ ]:
def plot_bioclim_variable_temperature(
    var_name,
    historic_ds,
    future_notip,
    periods
):
    period_keys = [f"{s}_{e}" for s, e in periods]
    col_titles = ["Historic", "2030–2044", "2050–2069", "2080–2099"]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=5,
        figsize=(19, 8),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [0.02, 1, 1, 1, 1]}
    )

    hist_var = historic_ds[var_name]

    # Row 1: absolute values
    row1_data = [hist_var] + [
        future_notip[key][var_name] for key in period_keys
    ]

    vmin = min(float(da.min()) for da in row1_data)
    vmax = max(float(da.max()) for da in row1_data)

    for j, da in enumerate(row1_data):
        da.plot(
            ax=axes[0, j + 1],
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            add_colorbar=False
        )
        axes[0, j + 1].set_title(col_titles[j], fontsize=16, fontweight="bold")

    sm1 = plt.cm.ScalarMappable(
        cmap="viridis",
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
    )
    cbar1 = fig.colorbar(
        sm1,
        ax=axes[0, 1:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"{var_name} [°C]")

    # Row 2: differences (future − historic)
    diffs = [
        future_notip[key][var_name] - hist_var
        for key in period_keys
    ]

    max_abs = max(float(np.abs(da).max()) for da in diffs)
    norm = mcolors.TwoSlopeNorm(
        vmin=-max_abs,
        vcenter=0.0,
        vmax=max_abs
    )

    # column 1 (historic stays empty)
    axes[1, 1].axis("off")

    for j, diff in enumerate(diffs):
        diff.plot(
            ax=axes[1, j + 2],  
            cmap="RdBu_r",
            norm=norm,
            add_colorbar=False
        )

    for ax in axes[1, 2:]:
        ax.set_title("")

    sm2 = plt.cm.ScalarMappable(
        cmap="RdBu_r",
        norm=norm
    )

    cbar2 = fig.colorbar(
        sm2,
        ax=axes[1, 2:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"Δ {var_name} [°C]"
    )


    # Row labels
    make_label_axis(axes[0, 0], var_name)
    make_label_axis(
        axes[1, 0],
        f"Δ {var_name}\nFuture − Historic"
    )

    cbar1.ax.tick_params(labelsize=12)
    cbar2.ax.tick_params(labelsize=12)

    cbar1.set_label(f"{var_name} [°C]", fontsize=14)
    cbar2.set_label(f"Δ {var_name} [°C]", fontsize=14)



    for ax in axes[:, 1:].flat:
        ax.set_xlabel("")
        ax.set_ylabel("")

    # Panel lettering follows the order used in the SI figure captions.
    for row, column, label in [
        (0, 1, "a"), (0, 2, "b"), (0, 3, "c"), (0, 4, "d"),
        (1, 2, "e"), (1, 3, "f"), (1, 4, "g"),
    ]:
        add_panel_label(axes[row, column], label)
        
    plt.savefig(
        os.path.join(PLOT_DIR, f"spatial_difference_across_time_periods_{var_name}.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
def plot_bioclim_variable_precipitation(
    var_name,
    historic_ds,
    future_notip,
    future_tip,
    periods
):
    period_keys = [f"{s}_{e}" for s, e in periods]
    col_titles = ["Historic", "2030–2044", "2050–2069", "2080–2099"]

    fig, axes = plt.subplots(
        nrows=5,
        ncols=5,
        figsize=(19, 18),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [0.02, 1, 1, 1, 1]}
    )

    hist_var = historic_ds[var_name]

    # Check if the variable is already in percentage
    is_percentage_var = var_name == "PS"

    # Row 1 & 3: absolute / percent values
    abs_data = (
        [hist_var] +
        [future_notip[k][var_name] for k in period_keys] +
        [future_tip[k][var_name] for k in period_keys]
    )

    abs_vmin = float(min(da.min() for da in abs_data))
    abs_vmax = float(max(da.max() for da in abs_data))
    abs_norm = mcolors.Normalize(vmin=abs_vmin, vmax=abs_vmax)

    abs_unit = "[%]" if is_percentage_var else "[mm]"


    # Rows 2 & 4: percentage difference vs historic
    perc_hist_diffs = (
        [percentage_difference(future_notip[k][var_name], hist_var)
         for k in period_keys] +
        [percentage_difference(future_tip[k][var_name], hist_var)
         for k in period_keys]
    )

    perc_hist_max = max(float(np.abs(da).max()) for da in perc_hist_diffs)
    perc_hist_norm = mcolors.TwoSlopeNorm(
        vmin=-perc_hist_max,
        vcenter=0.0,
        vmax=perc_hist_max
    )

    # Row 1: historic + no tipping (absolute/percent)
    row1_data = [hist_var] + [future_notip[k][var_name] for k in period_keys]

    for j, da in enumerate(row1_data):
        da.plot(
            ax=axes[0, j + 1],
            cmap="viridis",
            norm=abs_norm,
            add_colorbar=False
        )
        axes[0, j + 1].set_title(col_titles[j], fontsize=16, fontweight="bold")

    sm_abs = plt.cm.ScalarMappable(cmap="viridis", norm=abs_norm)
    cbar_abs1 = fig.colorbar(
        sm_abs,
        ax=axes[0, 1:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"{var_name} {abs_unit}"
    )

    # Row 2: no tipping vs historic (%)
    axes[1, 1].axis("off")
    for j, da in enumerate(perc_hist_diffs[:len(period_keys)]):
        da.plot(
            ax=axes[1, j + 2],
            cmap="BrBG",
            norm=perc_hist_norm,
            add_colorbar=False
        )

    sm_perc_hist = plt.cm.ScalarMappable(cmap="BrBG", norm=perc_hist_norm)
    cbar_perc1 = fig.colorbar(
        sm_perc_hist,
        ax=axes[1, 2:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"%Δ {var_name}\nNo tipping − Historic"
    )


    # Row 3: tipping absolute/percent
    axes[2, 1].axis("off")
    for j, da in enumerate([future_tip[k][var_name] for k in period_keys]):
        da.plot(
            ax=axes[2, j + 2],
            cmap="viridis",
            norm=abs_norm,
            add_colorbar=False
        )

    cbar_abs2 = fig.colorbar(
        sm_abs,
        ax=axes[2, 2:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"{var_name} {abs_unit}"
    )


    # Row 4: tipping vs historic (%)
    axes[3, 1].axis("off")
    for j, da in enumerate(perc_hist_diffs[len(period_keys):]):
        da.plot(
            ax=axes[3, j + 2],
            cmap="BrBG",
            norm=perc_hist_norm,
            add_colorbar=False
        )

    cbar_perc2 = fig.colorbar(
        sm_perc_hist,
        ax=axes[3, 2:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"%Δ {var_name}\nTipping − Historic"
    )


    # Row 5: tipping vs no tipping (%)
    diffs_tip_notip = [
        percentage_difference(future_tip[k][var_name], future_notip[k][var_name])
        for k in period_keys
    ]
    max_abs = max(float(np.abs(da).max()) for da in diffs_tip_notip)
    norm_tip_notip = mcolors.TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)

    axes[4, 1].axis("off")
    for j, da in enumerate(diffs_tip_notip):
        da.plot(
            ax=axes[4, j + 2],
            cmap="BrBG",
            norm=norm_tip_notip,
            add_colorbar=False
        )

    sm_tip_notip = plt.cm.ScalarMappable(cmap="BrBG", norm=norm_tip_notip)
    cbar_perc3 = fig.colorbar(
        sm_tip_notip,
        ax=axes[4, 2:],
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        label=f"%Δ {var_name}\nTipping − No tipping"
    )
    
    # Row labels
    make_label_axis(axes[0, 0], var_name)
    make_label_axis(axes[1, 0], "%Δ\nNo tipping − Historic")
    make_label_axis(axes[2, 0], f"{var_name}\nTipping")
    make_label_axis(axes[3, 0], "%Δ\nTipping − Historic")
    make_label_axis(axes[4, 0], "%Δ\nTipping − No tipping")

    # Remove duplicate titles
    for ax in axes[1:, 2:].flat:
        ax.set_title("")

    # Style colorbars
    for cbar in [cbar_abs1, cbar_abs2, cbar_perc1, cbar_perc2, cbar_perc3]:
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label(cbar.ax.get_ylabel(), fontsize=14)

    # Remove axis labels
    for ax in axes[:, 1:].flat:
        ax.set_xlabel("")
        ax.set_ylabel("")

    # Panel lettering follows the order used in the SI figure captions.
    for row, column, label in [
        (0, 1, "a"), (0, 2, "b"), (0, 3, "c"), (0, 4, "d"),
        (1, 2, "e"), (1, 3, "f"), (1, 4, "g"),
        (2, 2, "h"), (2, 3, "i"), (2, 4, "j"),
        (3, 2, "k"), (3, 3, "l"), (3, 4, "m"),
        (4, 2, "n"), (4, 3, "o"), (4, 4, "p"),
    ]:
        add_panel_label(axes[row, column], label)

    plt.savefig(
        os.path.join(PLOT_DIR, f"spatial_difference_across_time_periods_{var_name}.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
plot_bioclim_variable_temperature(
    var_name="MAT",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_temperature(
    var_name="TS",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)


In [ ]:
plot_bioclim_variable_temperature(
    var_name="MTWeQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)


In [ ]:
plot_bioclim_variable_temperature(
    var_name="MTDQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_temperature(
    var_name="MTWaQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_temperature(
    var_name="MTCQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="AP",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PWM",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PDM",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PWeQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PWaQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PCQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PS",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods,
)

In [ ]:
plot_bioclim_variable_precipitation(
    var_name="PDQ",
    historic_ds=historic_bioclim_amazon,
    future_notip=future_notip,
    future_tip=future_tip,
    periods=periods,
)